# Challenge 1 - Tic Tac Toe

In this lab you will perform deep learning analysis on a dataset of playing [Tic Tac Toe](https://en.wikipedia.org/wiki/Tic-tac-toe).

There are 9 grids in Tic Tac Toe that are coded as the following picture shows:

![Tic Tac Toe Grids](tttboard.jpg)

In the first 9 columns of the dataset you can find which marks (`x` or `o`) exist in the grids. If there is no mark in a certain grid, it is labeled as `b`. The last column is `class` which tells you whether Player X (who always moves first in Tic Tac Toe) wins in this configuration. Note that when `class` has the value `False`, it means either Player O wins the game or it ends up as a draw.

Follow the steps suggested below to conduct a neural network analysis using Tensorflow and Keras. You will build a deep learning model to predict whether Player X wins the game or not.

## Step 1: Data Engineering

This dataset is almost in the ready-to-use state so you do not need to worry about missing values and so on. Still, some simple data engineering is needed.

1. Read `tic-tac-toe.csv` into a dataframe.
1. Inspect the dataset. Determine if the dataset is reliable by eyeballing the data.
1. Convert the categorical values to numeric in all columns.
1. Separate the inputs and output.
1. Normalize the input data.

In [1]:
# your code here
import pandas as pd

df = pd.read_csv('tic-tac-toe.csv')
print(df.shape)
df.head()

(958, 10)


,TL,TM,TR,ML,MM,MR,BL,BM,BR,class
0,x,x,x,x,o,o,x,o,o,True
1,x,x,x,x,o,o,o,x,o,True
2,x,x,x,x,o,o,o,o,x,True
3,x,x,x,x,o,o,o,b,b,True
4,x,x,x,x,o,o,b,o,b,True


In [2]:
print(df.isna().sum().sum())
print(df['class'].value_counts())

0
class
True     626
False    332
Name: count, dtype: int64


In [3]:
board = {'x': 1, 'o': -1, 'b': 0}

X = df.drop(columns='class')
for col in X.columns:
    X[col] = X[col].map(board)

y = df['class'].astype(int)

print(X.head())
print(y.value_counts())

   TL  TM  TR  ML  MM  MR  BL  BM  BR
0   1   1   1   1  -1  -1   1  -1  -1
1   1   1   1   1  -1  -1  -1   1  -1
2   1   1   1   1  -1  -1  -1  -1   1
3   1   1   1   1  -1  -1  -1   0   0
4   1   1   1   1  -1  -1   0  -1   0
class
1    626
0    332
Name: count, dtype: int64


## Step 2: Build Neural Network

To build the neural network, you can refer to your own codes you wrote while following the [Deep Learning with Python, TensorFlow, and Keras tutorial](https://www.youtube.com/watch?v=wQ8BIBpya2k) in the lesson. It's pretty similar to what you will be doing in this lab.

1. Split the training and test data.
1. Create a `Sequential` model.
1. Add several layers to your model. Make sure you use ReLU as the activation function for the middle layers. Use Softmax for the output layer because each output has a single lable and all the label probabilities add up to 1.
1. Compile the model using `adam` as the optimizer and `sparse_categorical_crossentropy` as the loss function. For metrics, use `accuracy` for now.
1. Fit the training data.
1. Evaluate your neural network model with the test data.
1. Save your model as `tic-tac-toe.model`.

In [4]:
# your code here
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

(766, 9) (192, 9)


In [5]:
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense

keras.utils.set_random_seed(42)

model = Sequential()
model.add(Input(shape=(9,)))
model.add(Dense(32, activation='relu'))
model.add(Dense(16, activation='relu'))
model.add(Dense(2, activation='softmax'))

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 2)              │            34 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 882 (3.45 KB)

 Trainable params: 882 (3.45 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.fit(X_train, y_train, epochs=50, verbose=0)

loss, accuracy = model.evaluate(X_test, y_test)
print('Test loss:', loss)
print('Test accuracy:', accuracy)

model.save('tic-tac-toe.keras')

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9844 - loss: 0.0473 
Test loss: 0.047309864312410355
Test accuracy: 0.984375


## Step 3: Make Predictions

Now load your saved model and use it to make predictions on a few random rows in the test dataset. Check if the predictions are correct.

In [7]:
# your code here
from tensorflow.keras.models import load_model

saved_model = load_model('tic-tac-toe.keras')

sample = X_test.sample(5, random_state=1)
probs = saved_model.predict(sample)
predicted = probs.argmax(axis=1)

print('Predicted:', predicted)
print('Real:     ', y_test.loc[sample.index].values)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
Predicted: [1 1 0 0 0]
Real:      [1 1 0 0 0]


## Step 4: Improve Your Model

Did your model achieve low loss (<0.1) and high accuracy (>0.95)? If not, try to improve your model.

But how? There are so many things you can play with in Tensorflow and in the next challenge you'll learn about these things. But in this challenge, let's just do a few things to see if they will help.

* Add more layers to your model. If the data are complex you need more layers. But don't use more layers than you need. If adding more layers does not improve the model performance you don't need additional layers.
* Adjust the learning rate when you compile the model. This means you will create a custom `tf.keras.optimizers.Adam` instance where you specify the learning rate you want. Then pass the instance to `model.compile` as the optimizer.
    * `tf.keras.optimizers.Adam` [reference](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam).
    * Don't worry if you don't understand what the learning rate does. You'll learn about it in the next challenge.
* Adjust the number of epochs when you fit the training data to the model. Your model performance continues to improve as you train more epochs. But eventually it will reach the ceiling and the performance will stay the same.

In [8]:
# your code here
from tensorflow.keras.optimizers import Adam

def build_model(learning_rate):
    keras.utils.set_random_seed(42)
    model = Sequential()
    model.add(Input(shape=(9,)))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(16, activation='relu'))
    model.add(Dense(2, activation='softmax'))
    model.compile(optimizer=Adam(learning_rate=learning_rate),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

for lr in [0.0001, 0.001, 0.01]:
    model = build_model(lr)
    history = model.fit(X_train, y_train, epochs=50, validation_split=0.2, verbose=0)
    loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
    print(f'learning rate {lr}: train acc {history.history["accuracy"][-1]:.3f}, val acc {history.history["val_accuracy"][-1]:.3f}, test acc {accuracy:.3f}, test loss {loss:.3f}')

learning rate 0.0001: train acc 0.757, val acc 0.766, test acc 0.740, test loss 0.553
learning rate 0.001: train acc 1.000, val acc 0.987, test acc 0.979, test loss 0.071
learning rate 0.01: train acc 1.000, val acc 0.987, test acc 0.979, test loss 0.060


In [9]:
for ep in [10, 50, 100]:
    model = build_model(0.001)
    history = model.fit(X_train, y_train, epochs=ep, validation_split=0.2, verbose=0)
    loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
    print(f'epochs {ep}: train acc {history.history["accuracy"][-1]:.3f}, val acc {history.history["val_accuracy"][-1]:.3f}, test acc {accuracy:.3f}, test loss {loss:.3f}')

epochs 10: train acc 0.820, val acc 0.825, test acc 0.755, test loss 0.470
epochs 50: train acc 1.000, val acc 0.987, test acc 0.979, test loss 0.071
epochs 100: train acc 1.000, val acc 0.987, test acc 0.979, test loss 0.073


**Which approach(es) did you find helpful to improve your model performance?**

In [ ]:
# your answer here
# What I observed: testing different numbers of epochs shows whether there are too few epochs or whether the model starts overfitting.
# 10 epochs were too few (test accuracy 0.755), 50 epochs were enough (0.979), and 100 epochs did not help (the test loss even rose a little).
# The learning rate also mattered: 0.0001 was too slow (0.740), while 0.001 and 0.01 both reached about 0.98.
